In [1]:
##########################################################################
# IMPORTS
##########################################################################

from pathlib import Path
import json

import numpy as np
import pandas as pd

import geopandas as gpd

import rasterio

import matplotlib.pyplot as plt

from shapely.geometry import box

from rasterstats import zonal_stats

##########################################################################
# PARAMETERS
##########################################################################

GRID_SIZE = 1.5

MIN_CELL_COVERAGE = 0.50

##########################################################################
# INPUT FOLDERS
##########################################################################

ROOT_DIR = Path("..")

INDICES_DIR = ROOT_DIR / "indices_aligned"

BOUNDARY_PATH = ROOT_DIR / "aoi" / "crop_boundary.gpkg"

##########################################################################
# OUTPUT
##########################################################################

OUTPUT_DIR = ROOT_DIR / "grid_analysis"

CSV_DIR = OUTPUT_DIR / "csv"

GPKG_DIR = OUTPUT_DIR / "gpkg"

PNG_DIR = OUTPUT_DIR / "preview"

for folder in [

    OUTPUT_DIR,
    CSV_DIR,
    GPKG_DIR,
    PNG_DIR

]:

    folder.mkdir(

        parents=True,

        exist_ok=True

    )

##########################################################################
# LOAD BOUNDARY
##########################################################################

crop_boundary = gpd.read_file(

    BOUNDARY_PATH

)

##########################################################################
# CREATE GRID
##########################################################################

grid_path = GPKG_DIR / "grid_1p5m.gpkg"

if grid_path.exists():

    print("\nGrid already exists.")

    grid = gpd.read_file(

        grid_path

    )

else:

    print("\nCreating grid...")

    minx, miny, maxx, maxy = crop_boundary.total_bounds

    cells = []

    x = minx

    while x < maxx:

        y = miny

        while y < maxy:

            cells.append(

                box(

                    x,

                    y,

                    x + GRID_SIZE,

                    y + GRID_SIZE

                )

            )

            y += GRID_SIZE

        x += GRID_SIZE

    grid = gpd.GeoDataFrame(

        geometry=cells,

        crs=crop_boundary.crs

    )

    ######################################################################
    # INTERSECTION
    ######################################################################

    grid = gpd.overlay(

        grid,

        crop_boundary,

        how="intersection"

    )

    ######################################################################
    # COVERAGE
    ######################################################################

    full_area = GRID_SIZE * GRID_SIZE

    grid["coverage"] = (

        grid.area /

        full_area

    )

    grid = grid[

        grid.coverage >= MIN_CELL_COVERAGE

    ].copy()

    ######################################################################
    # SORT
    ######################################################################

    grid["cx"] = grid.centroid.x

    grid["cy"] = grid.centroid.y

    grid = grid.sort_values(

        by=[

            "cy",

            "cx"

        ],

        ascending=[

            False,

            True

        ]

    ).reset_index(

        drop=True

    )

    ######################################################################
    # IDS
    ######################################################################

    grid["cell_id"] = np.arange(

        1,

        len(grid)+1

    )

    grid["grid_size"] = GRID_SIZE

    grid["version"] = 1

    grid["area_m2"] = grid.area

    grid = grid.drop(

        columns=[

            "cx",

            "cy"

        ]

    )

    ######################################################################
    # SAVE GRID
    ######################################################################

    grid.to_file(

        grid_path,

        driver="GPKG"

    )

    grid.to_file(

        GPKG_DIR /

        "grid_1p5m.geojson",

        driver="GeoJSON"

    )

##########################################################################
# PREVIEW GRID
##########################################################################

preview = PNG_DIR / "grid_preview.png"

fig, ax = plt.subplots(

    figsize=(8,10)

)

crop_boundary.boundary.plot(

    ax=ax,

    color="black",

    linewidth=2

)

grid.plot(

    ax=ax,

    edgecolor="red",

    facecolor="none",

    linewidth=0.3

)

ax.set_axis_off()

plt.tight_layout()

plt.savefig(

    preview,

    dpi=300,

    bbox_inches="tight"

)

plt.close()

##########################################################################
# FIND NEW FLIGHTS
##########################################################################

csv_path = CSV_DIR / "cell_time_series.csv"

if csv_path.exists():

    existing_database = pd.read_csv(

        csv_path

    )

    processed_flights = set(

        existing_database.flight.unique()

    )

else:

    existing_database = pd.DataFrame()

    processed_flights = set()

##########################################################################

new_flights = []

years = sorted([

    folder

    for folder in INDICES_DIR.iterdir()

    if folder.is_dir()

])

for year_folder in years:

    year = year_folder.name

    flights = sorted([

        f

        for f in year_folder.iterdir()

        if f.is_dir()

    ])

    for flight in flights:

        if flight.name in processed_flights:

            print(

                f"SKIP -> {flight.name}"

            )

            continue

        new_flights.append(

            (

                year,

                flight

            )

        )

##########################################################################

print()

print("="*60)

print(

    f"Grid cells : {len(grid)}"

)

print(

    f"Flights to process : {len(new_flights)}"

)

print("="*60)

for year, flight in new_flights:

    print(

        year,

        flight.name

    )


Grid already exists.
SKIP -> flight_2025_04_09
SKIP -> flight_2025_05_08
SKIP -> flight_2025_05_14
SKIP -> flight_2025_05_19
SKIP -> flight_2025_06_03
SKIP -> flight_2025_06_12
SKIP -> flight_2025_06_19
SKIP -> flight_2025_07_04
SKIP -> flight_2025_07_08
SKIP -> flight_2025_08_11
SKIP -> flight_2025_08_19
SKIP -> flight_2026_04_01
SKIP -> flight_2026_04_07
SKIP -> flight_2026_04_15
SKIP -> flight_2026_04_22
SKIP -> flight_2026_05_01
SKIP -> flight_2026_05_22

Grid cells : 1021
Flights to process : 3
2026 flight_2026_05_28
2026 flight_2026_06_01
2026 flight_2026_06_19


In [2]:
##########################################################################
# PROCESS ONE FLIGHT
##########################################################################

def process_flight(year, flight):

    print("\n" + "="*70)
    print(f"PROCESSING {flight.name}")
    print("="*70)

    ######################################################################
    # DATE
    ######################################################################

    date = flight.name.replace(
        "flight_",
        ""
    )

    ######################################################################
    # INPUT FILES
    ######################################################################

    ndvi_path = flight / f"{date}_NDVI.tif"

    gndvi_path = flight / f"{date}_GNDVI.tif"

    ndre_path = flight / f"{date}_NDRE.tif"

    temperature_path = flight / f"{date}_canopy_temperature.tif"

    ######################################################################
    # CHECK FILES
    ######################################################################

    required_files = [

        ndvi_path,

        gndvi_path,

        ndre_path,

        temperature_path

    ]

    missing = [

        file.name

        for file in required_files

        if not file.exists()

    ]

    if len(missing) > 0:

        print("\nMissing files:")

        for file in missing:

            print(f"   - {file}")

        return

    ######################################################################
    # OUTPUT FOLDER
    ######################################################################

    output_year = OUTPUT_DIR / year

    output_year.mkdir(

        exist_ok=True

    )

    output_flight = output_year / flight.name

    output_flight.mkdir(

        exist_ok=True

    )

    ######################################################################
    # OUTPUT FILES
    ######################################################################

    stress_gpkg = (

        output_flight /

        f"{date}_stress_grid_1p5m.gpkg"

    )

    stress_tif = (

        output_flight /

        f"{date}_stress_score_1p5m.tif"

    )

    preview_png = (

        output_flight /

        f"{date}_preview.png"

    )

    threshold_json = (

        output_flight /

        f"{date}_thresholds.json"

    )

    ######################################################################
    # ZONAL STATISTICS
    ######################################################################

    print(

        "\nCalculating zonal statistics..."

    )

    ndvi_stats = zonal_stats(

        grid,

        ndvi_path,

        stats="mean",

        geojson_out=False

    )

    gndvi_stats = zonal_stats(

        grid,

        gndvi_path,

        stats="mean",

        geojson_out=False

    )

    ndre_stats = zonal_stats(

        grid,

        ndre_path,

        stats="mean",

        geojson_out=False

    )

    temperature_stats = zonal_stats(

        grid,

        temperature_path,

        stats="mean",

        geojson_out=False

    )

    ######################################################################
    # BUILD GEODATAFRAME
    ######################################################################

    flight_df = grid.copy()

    flight_df["year"] = year

    flight_df["date"] = date

    flight_df["flight"] = flight.name

    flight_df["NDVI"] = [

        s["mean"]

        for s in ndvi_stats

    ]

    flight_df["GNDVI"] = [

        s["mean"]

        for s in gndvi_stats

    ]

    flight_df["NDRE"] = [

        s["mean"]

        for s in ndre_stats

    ]

    flight_df["Temperature"] = [

        s["mean"]

        for s in temperature_stats

    ]

    ######################################################################
    # REMOVE EMPTY CELLS
    ######################################################################

    flight_df = flight_df.dropna(

        subset=[

            "NDVI",

            "GNDVI",

            "NDRE",

            "Temperature"

        ]

    ).copy()

    print(

        f"Valid cells: {len(flight_df)}"

    )
        ######################################################################
    # CALCULATE STRESS SCORE
    ######################################################################

    print("\nCalculating stress score...")

    # --------------------------------------------------
    # Percentiles
    # --------------------------------------------------

    ndvi_p20 = flight_df["NDVI"].quantile(0.20)
    ndvi_p40 = flight_df["NDVI"].quantile(0.40)

    gndvi_p20 = flight_df["GNDVI"].quantile(0.20)
    gndvi_p40 = flight_df["GNDVI"].quantile(0.40)

    ndre_p20 = flight_df["NDRE"].quantile(0.20)
    ndre_p40 = flight_df["NDRE"].quantile(0.40)

    temp_p70 = flight_df["Temperature"].quantile(0.70)
    temp_p85 = flight_df["Temperature"].quantile(0.85)

    ######################################################################
    # SCORE FUNCTIONS
    ######################################################################

    def vegetation_score(values, p20, p40):

        score = np.zeros(len(values))

        score[values < p20] = 1

        score[
            (values >= p20) &
            (values < p40)
        ] = 0.5

        return score

    ######################################################################

    def temperature_score(values, p70, p85):

        score = np.zeros(len(values))

        score[
            (values >= p70) &
            (values < p85)
        ] = 0.5

        score[
            values >= p85
        ] = 1

        return score

    ######################################################################
    # CALCULATE SCORES
    ######################################################################

    flight_df["NDVI_score"] = vegetation_score(

        flight_df["NDVI"].values,

        ndvi_p20,

        ndvi_p40

    )

    flight_df["GNDVI_score"] = vegetation_score(

        flight_df["GNDVI"].values,

        gndvi_p20,

        gndvi_p40

    )

    flight_df["NDRE_score"] = vegetation_score(

        flight_df["NDRE"].values,

        ndre_p20,

        ndre_p40

    )

    flight_df["Temperature_score"] = temperature_score(

        flight_df["Temperature"].values,

        temp_p70,

        temp_p85

    )

    ######################################################################
    # FINAL STRESS
    ######################################################################

    flight_df["Stress"] = (

        flight_df["NDVI_score"]

        +

        flight_df["GNDVI_score"]

        +

        flight_df["NDRE_score"]

        +

        flight_df["Temperature_score"]

    )

    ######################################################################
    # SAVE THRESHOLDS
    ######################################################################

    thresholds = {

        "NDVI_P20": float(ndvi_p20),
        "NDVI_P40": float(ndvi_p40),

        "GNDVI_P20": float(gndvi_p20),
        "GNDVI_P40": float(gndvi_p40),

        "NDRE_P20": float(ndre_p20),
        "NDRE_P40": float(ndre_p40),

        "TEMP_P70": float(temp_p70),
        "TEMP_P85": float(temp_p85)

    }

    with open(

        threshold_json,

        "w"

    ) as f:

        json.dump(

            thresholds,

            f,

            indent=4

        )

    ######################################################################
    # SAVE GEOPACKAGE
    ######################################################################

    flight_df.to_file(

        stress_gpkg,

        driver="GPKG"

    )

    ######################################################################
    # UPDATE DATABASE
    ######################################################################

    csv_columns = [

        "year",

        "date",

        "flight",

        "cell_id",

        "NDVI",

        "GNDVI",

        "NDRE",

        "Temperature",

        "Stress"

    ]

    new_rows = flight_df[

        csv_columns

    ].copy()

    if csv_path.exists():

        database = pd.read_csv(

            csv_path

        )

        database = pd.concat(

            [

                database,

                new_rows

            ],

            ignore_index=True

        )

    else:

        database = new_rows

    database.to_csv(

        csv_path,

        index=False

    )

    ######################################################################
    # SUMMARY
    ######################################################################

    print()

    print(f"GeoPackage saved : {stress_gpkg.name}")

    print(f"Database rows    : {len(database)}")

    print(f"Threshold file   : {threshold_json.name}")

    print(f"Mean stress      : {flight_df['Stress'].mean():.2f}")

    ######################################################################
    # PREVIEW
    ######################################################################

    print("\nCreating preview...")

    fig, ax = plt.subplots(

        figsize=(8,10)

    )

    ######################################################################
    # PLOT
    ######################################################################

    flight_df.plot(

        column="Stress",

        ax=ax,

        cmap="RdYlGn_r",

        vmin=0,

        vmax=4,

        linewidth=0.25,

        edgecolor="black",

        legend=True,

        legend_kwds={

            "label":"Stress Score",

            "shrink":0.75

        }

    )

    crop_boundary.boundary.plot(

        ax=ax,

        color="black",

        linewidth=1.5

    )

    ax.set_axis_off()

    ######################################################################
    # TITLE
    ######################################################################

    ax.set_title(

        f"Stress Map\n{date}",

        fontsize=16,

        weight="bold"

    )

    ######################################################################
    # INFORMATION BOX
    ######################################################################

    info = (

        f"Cells : {len(flight_df)}\n"

        f"Mean Stress : {flight_df['Stress'].mean():.2f}\n\n"

        f"NDVI   P20={ndvi_p20:.3f}   P40={ndvi_p40:.3f}\n"

        f"GNDVI  P20={gndvi_p20:.3f}   P40={gndvi_p40:.3f}\n"

        f"NDRE   P20={ndre_p20:.3f}   P40={ndre_p40:.3f}\n"

        f"TEMP   P70={temp_p70:.2f}°C   P85={temp_p85:.2f}°C"

    )

    ax.text(

        1.02,

        0.98,

        info,

        transform=ax.transAxes,

        va="top",

        fontsize=9,

        bbox=dict(

            facecolor="white",

            alpha=0.85,

            edgecolor="gray"

        )

    )

    ######################################################################
    # SAVE
    ######################################################################

    plt.tight_layout()

    plt.savefig(

        preview_png,

        dpi=300,

        bbox_inches="tight"

    )

    plt.close()

    ######################################################################
    # END
    ######################################################################

    print("\nFinished successfully.")

    print(f"Preview     : {preview_png.name}")

    print(f"GeoPackage  : {stress_gpkg.name}")

    print(f"CSV updated : {csv_path.name}")

    print("="*70)

In [3]:
##########################################################################
# MAIN
##########################################################################

for year, flight in new_flights:

    process_flight(

        year,

        flight

    )



PROCESSING flight_2026_05_28

Calculating zonal statistics...
Valid cells: 1021

Calculating stress score...

GeoPackage saved : 2026_05_28_stress_grid_1p5m.gpkg
Database rows    : 18372
Threshold file   : 2026_05_28_thresholds.json
Mean stress      : 1.12

Creating preview...

Finished successfully.
Preview     : 2026_05_28_preview.png
GeoPackage  : 2026_05_28_stress_grid_1p5m.gpkg
CSV updated : cell_time_series.csv

PROCESSING flight_2026_06_01

Calculating zonal statistics...
Valid cells: 1021

Calculating stress score...

GeoPackage saved : 2026_06_01_stress_grid_1p5m.gpkg
Database rows    : 19393
Threshold file   : 2026_06_01_thresholds.json
Mean stress      : 1.12

Creating preview...

Finished successfully.
Preview     : 2026_06_01_preview.png
GeoPackage  : 2026_06_01_stress_grid_1p5m.gpkg
CSV updated : cell_time_series.csv

PROCESSING flight_2026_06_19

Calculating zonal statistics...
Valid cells: 1021

Calculating stress score...

GeoPackage saved : 2026_06_19_stress_grid_1p5